# Threshold Function Enumeration: A Complete Implementation

This notebook provides clean, well-documented implementations for enumerating
Linear Threshold Functions (LTFs). It consolidates various approaches explored
during research.

## Background

A **threshold function** (or Linear Threshold Function) is a Boolean function
f: {0,1}ⁿ → {0,1} that can be written as:

$$f(x) = \begin{cases} 1 & \text{if } \sum_{i=1}^n w_i x_i \geq T \\ 0 & \text{otherwise} \end{cases}$$

The count N(n) of threshold functions on n variables is a famous sequence (OEIS A000609).

## Mathematical Context

- **Hyperplane Arrangements**: The weight space ℝⁿ is partitioned by hyperplanes
  of the form w·(x-y) = 0 for vertices x,y ∈ {0,1}ⁿ. Each region (chamber)
  corresponds to a unique ordering of weighted sums.

- **Hyperoctahedral Group B_n**: The symmetry group of the n-hypercube, with
  order |B_n| = 2ⁿ × n!. Used for symmetry reduction in enumeration.

- **Zaslavsky's Theorem**: Counts regions in hyperplane arrangements using
  the Möbius function of the intersection lattice.

## Literature

Key references:
- Muroga, S. (1971). Threshold Logic and Its Applications
- Winder, R.O. (1966). Enumeration of Seven-Argument Threshold Functions
- Zuev, Y.A. (1989). Asymptotics of the logarithm of the number of threshold functions

In [1]:
import itertools
import numpy as np
import math
import time
from collections import defaultdict, Counter
from z3 import *

# Known values (OEIS A000609)
KNOWN_VALUES = {
    0: 2,
    1: 4,
    2: 14,
    3: 104,
    4: 1882,
    5: 94572,
    6: 15028134,
    7: 8378070864,
    8: 17561539552946,
    9: 144130531453121108,
}

print("Known values N(n) - Number of threshold functions on n variables:")
print("=" * 50)
for n, v in list(KNOWN_VALUES.items())[:8]:
    ratio = v / KNOWN_VALUES[n-1] if n > 0 else 1
    print(f"  N({n}) = {v:>20,}   (×{ratio:,.1f})")

Known values N(n) - Number of threshold functions on n variables:
  N(0) =                    2   (×1.0)
  N(1) =                    4   (×2.0)
  N(2) =                   14   (×3.5)
  N(3) =                  104   (×7.4)
  N(4) =                1,882   (×18.1)
  N(5) =               94,572   (×50.3)
  N(6) =           15,028,134   (×158.9)
  N(7) =        8,378,070,864   (×557.5)


---
## Method 1: Symmetry-Breaking Enumeration (Fastest)

The most efficient method uses:
1. Z3 SMT solver with biconditional encoding
2. Symmetry-breaking constraints: w₀ ≥ w₁ ≥ ... ≥ wₙ₋₁ ≥ 0
3. Orbit size computation under B_n

This finds canonical representatives and sums orbit sizes.

In [2]:
class HyperoctahedralGroup:
    """
    The hyperoctahedral group B_n for computing orbit sizes.
    
    B_n is the symmetry group of the n-hypercube.
    Elements: pairs (permutation, signs) where
    - permutation ∈ S_n shuffles coordinates
    - signs ∈ {±1}ⁿ flips coordinates (x_i → 1-x_i)
    
    Order: |B_n| = 2ⁿ × n!
    """
    
    def __init__(self, n):
        self.n = n
        self.order = (2 ** n) * math.factorial(n)
        self.num_vertices = 2 ** n
        
        # Precompute vertices
        self.vertices = np.array(list(itertools.product((0, 1), repeat=n)), dtype=np.int8)
        self.vertex_to_idx = {tuple(v): i for i, v in enumerate(self.vertices)}
        
        # Precompute permutation actions for fast orbit computation
        self._precompute_permutations()
    
    def _precompute_permutations(self):
        """Precompute how each group element permutes vertex indices."""
        perms = list(itertools.permutations(range(self.n)))
        sign_patterns = list(itertools.product([0, 1], repeat=self.n))
        
        self.permutations = np.zeros((self.order, self.num_vertices), dtype=np.int32)
        
        g_idx = 0
        for perm in perms:
            for signs in sign_patterns:
                for v_idx in range(self.num_vertices):
                    v = self.vertices[v_idx]
                    flipped = np.where(signs, 1 - v, v)
                    permuted = flipped[list(perm)]
                    self.permutations[g_idx, v_idx] = self.vertex_to_idx[tuple(permuted)]
                g_idx += 1
    
    def orbit_size(self, func_array):
        """Compute orbit size using vectorized operations."""
        transformed = func_array[self.permutations]
        unique = np.unique(transformed, axis=0)
        return len(unique)


def count_threshold_functions(n, verbose=True, progress_interval=100):
    """
    Count threshold functions using symmetry-breaking Z3 enumeration.
    
    This is the fastest exact method for moderate n.
    
    Args:
        n: Number of variables
        verbose: Print progress
        progress_interval: Update frequency
    
    Returns:
        (total_count, num_representatives, elapsed_time)
    """
    vertices = list(itertools.product((0, 1), repeat=n))
    num_vertices = 2 ** n
    
    # Z3 variables
    f = [Bool(f'f_{i}') for i in range(num_vertices)]
    w = [Real(f'w_{i}') for i in range(n)]
    T = Real('T')
    
    solver = Solver()
    
    # Core constraint: f[i] ↔ (weighted_sum ≥ T)
    for i, v in enumerate(vertices):
        weighted_sum = sum(w[k] * v[k] for k in range(n))
        solver.add(f[i] == (weighted_sum >= T))
    
    # Symmetry-breaking: w[0] ≥ w[1] ≥ ... ≥ w[n-1] ≥ 0
    for i in range(n - 1):
        solver.add(w[i] >= w[i + 1])
    solver.add(w[n - 1] >= 0)
    
    # Initialize symmetry group for orbit computation
    Bn = HyperoctahedralGroup(n)
    
    found_functions = set()
    total_count = 0
    num_reps = 0
    
    start_time = time.time()
    
    if verbose:
        print(f"Enumerating N({n})...")
        print(f"Expected: {KNOWN_VALUES.get(n, '?'):,}")
        print("-" * 50)
    
    while solver.check() == sat:
        model = solver.model()
        func_tuple = tuple(is_true(model.eval(f[i], model_completion=True)) 
                          for i in range(num_vertices))
        
        if func_tuple in found_functions:
            break
        
        found_functions.add(func_tuple)
        num_reps += 1
        
        # Compute orbit size
        func_array = np.array(func_tuple, dtype=np.int8)
        orbit_sz = Bn.orbit_size(func_array)
        total_count += orbit_sz
        
        if verbose and num_reps % progress_interval == 0:
            elapsed = time.time() - start_time
            expected = KNOWN_VALUES.get(n, total_count * 2)
            pct = 100 * total_count / expected if expected > 0 else 0
            print(f"  {num_reps:,} reps → {total_count:,} functions ({pct:.1f}%) [{elapsed:.1f}s]")
        
        # Block this solution
        solver.add(Or([f[i] != model.eval(f[i], model_completion=True) 
                       for i in range(num_vertices)]))
    
    elapsed_time = time.time() - start_time
    
    if verbose:
        print("-" * 50)
        print(f"Completed in {elapsed_time:.2f}s")
    
    return total_count, num_reps, elapsed_time

In [3]:
# Verify the implementation
print("=" * 60)
print("VERIFICATION")
print("=" * 60)

results = []
for n in range(1, 5):
    count, reps, elapsed = count_threshold_functions(n, verbose=False)
    expected = KNOWN_VALUES[n]
    status = "✓" if count == expected else "✗"
    results.append((n, count, reps, elapsed, status))
    print(f"N({n}) = {count:,} (expected {expected:,}) {status}  [{reps} reps, {elapsed:.3f}s]")

print("\n" + "=" * 60)
print("All verifications passed!" if all(r[4] == "✓" for r in results) else "Some verifications failed!")

VERIFICATION
N(1) = 4 (expected 4) ✓  [3 reps, 0.016s]
N(2) = 14 (expected 14) ✓  [5 reps, 0.003s]
N(3) = 104 (expected 104) ✓  [10 reps, 0.007s]
N(4) = 1,882 (expected 1,882) ✓  [27 reps, 0.041s]

All verifications passed!


---
## Method 2: Chamber Enumeration (Activation Axis)

This method enumerates "chambers" (regions in weight space) and
derives threshold functions from the orderings they induce.

Each chamber corresponds to a unique ordering of weighted sums.
Threshold functions are "cuts" in these orderings.

In [4]:
def enumerate_chambers(n):
    """
    Enumerate chambers (valid sign patterns) in the hyperplane arrangement.
    
    A sign pattern assigns σ(d) ∈ {-1, +1} to each difference vector d = x - y.
    A pattern is valid if there exists W such that sign(W·d) matches for all d.
    """
    from scipy.optimize import linprog
    
    vertices = list(itertools.product((0, 1), repeat=n))
    
    # Get normalized difference vectors
    diff_vectors = []
    for i, x in enumerate(vertices):
        for j, y in enumerate(vertices):
            if i < j:
                d = tuple(a - b for a, b in zip(x, y))
                neg_d = tuple(-c for c in d)
                if d not in [dv for dv in diff_vectors] and neg_d not in [dv for dv in diff_vectors]:
                    diff_vectors.append(d)
    
    num_diffs = len(diff_vectors)
    
    if num_diffs > 20:
        print(f"Too many difference vectors ({num_diffs}) for brute force")
        return None
    
    valid_chambers = []
    epsilon = 1e-6
    
    for pattern_bits in range(2 ** num_diffs):
        signs = [2 * ((pattern_bits >> i) & 1) - 1 for i in range(num_diffs)]
        
        # Check LP feasibility
        A_ub = [[-signs[i] * d[k] for k in range(n)] for i, d in enumerate(diff_vectors)]
        b_ub = [-epsilon] * num_diffs
        bounds = [(-1, 1)] * n
        
        result = linprog([0] * n, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
        
        if result.success:
            valid_chambers.append((tuple(signs), result.x))
    
    return valid_chambers, diff_vectors


def chamber_to_functions(n, chamber, verbose=False):
    """
    Given a chamber (sign pattern + witness W), enumerate threshold functions.
    
    The ordering of weighted sums allows 2^n + 1 possible threshold placements.
    """
    vertices = list(itertools.product((0, 1), repeat=n))
    num_v = len(vertices)
    
    signs, W = chamber
    
    # Compute and sort weighted sums
    sums = [(sum(W[k] * v[k] for k in range(n)), i, v) for i, v in enumerate(vertices)]
    sums.sort()
    
    # Generate functions by placing thresholds
    functions = set()
    
    # All False
    functions.add(tuple(False for _ in range(num_v)))
    
    # Cut after each position
    for cut in range(num_v):
        func_list = [False] * num_v
        for i in range(cut + 1, num_v):
            func_list[sums[i][1]] = True
        functions.add(tuple(func_list))
    
    # All True
    functions.add(tuple(True for _ in range(num_v)))
    
    return functions


def count_via_chambers(n):
    """Count threshold functions via chamber enumeration."""
    result = enumerate_chambers(n)
    if result is None:
        return None
    
    chambers, diff_vectors = result
    
    all_functions = set()
    for chamber in chambers:
        funcs = chamber_to_functions(n, chamber)
        all_functions.update(funcs)
    
    return len(all_functions), len(chambers)


print("=" * 60)
print("CHAMBER ENUMERATION METHOD")
print("=" * 60)

for n in range(1, 4):
    count, num_chambers = count_via_chambers(n)
    expected = KNOWN_VALUES[n]
    status = "✓" if count == expected else "✗"
    print(f"n={n}: {num_chambers} chambers → {count} functions (expected {expected}) {status}")

CHAMBER ENUMERATION METHOD
n=1: 2 chambers → 4 functions (expected 4) ✓
n=2: 8 chambers → 14 functions (expected 14) ✓
n=3: 96 chambers → 104 functions (expected 104) ✓


---
## Analysis: Chamber-Function Relationship

Key insight: Functions can appear in multiple chambers (boundary functions).
Understanding this "overlap" is central to the counting problem.

In [5]:
def analyze_overlap(n):
    """Analyze how functions are distributed across chambers."""
    result = enumerate_chambers(n)
    if result is None:
        return
    
    chambers, _ = result
    
    # Count chambers per function
    func_chamber_count = defaultdict(int)
    for chamber in chambers:
        for f in chamber_to_functions(n, chamber):
            func_chamber_count[f] += 1
    
    distribution = Counter(func_chamber_count.values())
    
    print(f"n = {n}: Function overlap distribution")
    print(f"{'Chambers':>10} | {'Functions':>10}")
    print("-" * 25)
    for num_ch, num_funcs in sorted(distribution.items()):
        print(f"{num_ch:>10} | {num_funcs:>10}")
    print(f"Total: {sum(distribution.values())} functions")
    
    # Compute average overlap
    total_instances = sum(len(chamber_to_functions(n, c)) for c in chambers)
    avg_overlap = total_instances / len(func_chamber_count)
    print(f"Average overlap: {avg_overlap:.2f}")

print("=" * 60)
print("OVERLAP ANALYSIS")
print("=" * 60)

for n in range(1, 4):
    analyze_overlap(n)
    print()

OVERLAP ANALYSIS
n = 1: Function overlap distribution
  Chambers |  Functions
-------------------------
         1 |          2
         2 |          2
Total: 4 functions
Average overlap: 1.50

n = 2: Function overlap distribution
  Chambers |  Functions
-------------------------
         2 |         12
         8 |          2
Total: 14 functions
Average overlap: 2.86

n = 3: Function overlap distribution
  Chambers |  Functions
-------------------------
         4 |         48
         6 |          8
         8 |         30
        12 |         16
        96 |          2
Total: 104 functions
Average overlap: 8.31



---
## Compute N(5)

Now let's use the fastest method to compute N(5) = 94,572.

In [6]:
print("=" * 60)
print("COMPUTING N(5)")
print("=" * 60)

count_5, reps_5, time_5 = count_threshold_functions(5, verbose=True, progress_interval=50)

print(f"\n{'='*60}")
print(f"RESULT: N(5) = {count_5:,}")
print(f"Expected:      {KNOWN_VALUES[5]:,}")
print(f"Match: {'✓ CORRECT!' if count_5 == KNOWN_VALUES[5] else '✗ WRONG'}")
print(f"Representatives: {reps_5:,}")
print(f"Time: {time_5:.2f}s")
print("=" * 60)

COMPUTING N(5)
Enumerating N(5)...
Expected: 94,572
--------------------------------------------------
  50 reps → 37,155 functions (39.3%) [0.5s]
  100 reps → 85,947 functions (90.9%) [1.0s]
--------------------------------------------------
Completed in 1.21s

RESULT: N(5) = 94,572
Expected:      94,572
Match: ✓ CORRECT!
Representatives: 119
Time: 1.21s


In [8]:
print("=" * 60)
print("COMPUTING N(6)")
print("=" * 60)

count_6, reps_6, time_6 = count_threshold_functions(6, verbose=True, progress_interval=50)

print(f"\n{'='*60}")
print(f"RESULT: N(6) = {count_6:,}")
print(f"Expected:      {KNOWN_VALUES[6]:,}")
print(f"Match: {'✓ CORRECT!' if count_6 == KNOWN_VALUES[6] else '✗ WRONG'}")
print(f"Representatives: {reps_6:,}")
print(f"Time: {time_6:.2f}s")
print("=" * 60)

COMPUTING N(6)
Enumerating N(6)...
Expected: 15,028,134
--------------------------------------------------
  50 reps → 276,465 functions (1.8%) [13.2s]
  100 reps → 896,685 functions (6.0%) [23.4s]
  150 reps → 1,457,517 functions (9.7%) [34.9s]
  200 reps → 2,229,757 functions (14.8%) [44.7s]
  250 reps → 3,245,917 functions (21.6%) [54.3s]
  300 reps → 3,723,709 functions (24.8%) [65.2s]
  350 reps → 4,340,989 functions (28.9%) [74.3s]
  400 reps → 5,191,549 functions (34.5%) [83.6s]
  450 reps → 5,881,789 functions (39.1%) [93.2s]
  500 reps → 6,732,349 functions (44.8%) [101.4s]
  550 reps → 7,824,829 functions (52.1%) [109.6s]
  600 reps → 8,734,749 functions (58.1%) [118.3s]
  650 reps → 9,607,901 functions (63.9%) [126.9s]
  700 reps → 10,037,417 functions (66.8%) [136.7s]
  750 reps → 10,566,857 functions (70.3%) [146.6s]
  800 reps → 11,375,945 functions (75.7%) [157.3s]
  850 reps → 12,164,105 functions (80.9%) [166.3s]
  900 reps → 12,967,625 functions (86.3%) [175.0s]
  950

---
## Summary Table

In [10]:
print("=" * 70)
print("SUMMARY: Threshold Function Counts")
print("=" * 70)
print(f"{'n':>3} | {'N(n)':>15} | {'Representatives':>15} | {'|B_n|':>10} | {'Avg Orbit':>10}")
print("-" * 70)

for n in range(1, 7):
    count, reps, _ = count_threshold_functions(n, verbose=False)
    Bn_order = (2 ** n) * math.factorial(n)
    avg_orbit = count / reps if reps > 0 else 0
    print(f"{n:>3} | {count:>15,} | {reps:>15,} | {Bn_order:>10,} | {avg_orbit:>10.1f}")

print("-" * 70)
print("\nNotes:")
print("  - N(n): Total number of threshold functions")
print("  - Representatives: Canonical forms under B_n symmetry")
print("  - |B_n|: Order of hyperoctahedral group = 2^n × n!")
print("  - Avg Orbit: Average orbit size = N(n) / Representatives")

SUMMARY: Threshold Function Counts
  n |            N(n) | Representatives |      |B_n| |  Avg Orbit
----------------------------------------------------------------------
  1 |               4 |               3 |          2 |        1.3
  2 |              14 |               5 |          8 |        2.8
  3 |             104 |              10 |         48 |       10.4
  4 |           1,882 |              27 |        384 |       69.7
  5 |          94,572 |             119 |      3,840 |      794.7
  6 |      15,028,134 |           1,113 |     46,080 |    13502.4
----------------------------------------------------------------------

Notes:
  - N(n): Total number of threshold functions
  - Representatives: Canonical forms under B_n symmetry
  - |B_n|: Order of hyperoctahedral group = 2^n × n!
  - Avg Orbit: Average orbit size = N(n) / Representatives


---
## References

1. **OEIS A000609** - Number of threshold functions of n or fewer variables
   https://oeis.org/A000609

2. **Muroga, S.** (1971). *Threshold Logic and Its Applications*. Wiley.

3. **Winder, R.O.** (1966). Enumeration of Seven-Argument Threshold Functions.
   *IEEE Transactions on Electronic Computers*, EC-15(3), 315-325.

4. **Zuev, Y.A.** (1989). Asymptotics of the logarithm of the number of threshold functions.
   *Soviet Mathematics Doklady*, 39(3), 512-513.

5. **Zaslavsky, T.** (1975). Facing up to Arrangements: Face-Count Formulas for
   Partitions of Space by Hyperplanes. *Memoirs of the AMS*, 154.